In [1]:
%pip install ir_datasets pandas nltk matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

# Lab 2: Models for the Cranfield Test Collection

---


### Task 1: Cranfield Experiments: Documents, queries and relevance assessments

The following code loads all data from the Cranfield experiments. 
Make sure you understand the contents of these tables.

For the relevance scores, note that:

| Score (`relevance`) | Original Definition / Meaning | Judgment Level | Total Judgments Count | Percentage |
| :---: | :--- | :--- | :---: | :---: |
| **`4`** | References which are a **complete answer** to the question. | Highest Relevance | 363 | 19.8% |
| **`3`** | References of a **high degree of relevance**, the lack of which either would have made the research impracticable or resulted in considerable extra work. | High Relevance | 734 | 40.0% |
| **`2`** | References which were **useful**, either as general background to the work or as suggesting methods of tackling certain aspects. | Useful / Background | 387 | 21.1% |
| **`1`** | References of **minimum interest**, for example, those included from an historical viewpoint. | Marginal Relevance | 128 | 7.0% |
| **`-1`** | References of **no interest** (explicitly marked non-relevant). | Not Relevant | 225 | 12.2% |

In [2]:
import ir_datasets
import pandas as pd

# Load the Cranfield dataset
dataset = ir_datasets.load('cranfield')

# 1. Documents DataFrame
docs_df = pd.DataFrame([doc._asdict() for doc in dataset.docs_iter()])

# 2. Queries DataFrame
queries_df = pd.DataFrame([query._asdict() for query in dataset.queries_iter()])

# 3. Relevance Assessments (Qrels) DataFrame
qrels_df = pd.DataFrame([qrel._asdict() for qrel in dataset.qrels_iter()])

# Inspect the outputs
print(f"Loaded {len(docs_df)} documents, {len(queries_df)} queries, and {len(qrels_df)} relevance assessments.\n")


[INFO] [starting] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[INFO] [finished] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz: [00:00] [50
                                                                                

Loaded 1400 documents, 225 queries, and 1837 relevance assessments.




### Task 2: Boolean model

Implement the Boolean model on the Cranfield data. That means, find boolean vector representations of documents and queries found in the Cranfield data.
Then, implement a search mechanism (find relevant documents for each query) and figure out a way to evaluate the performance of this model (use existing relevance scores in qrels_df).

See how performance varies if you change preprocessing options (from last week's lab)

In [ ]:
import nltk
from nltk.stem.snowball import PorterStemmer
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer as wnl

stemmer = PorterStemmer()

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("punkt_tab", quiet=True)

stop_words = set(stopwords.words("english"))

tokens = docs_df.text.apply(nltk.word_tokenize)

tokens.head(10)

A = {}  # only tokenize
B = {}  # tokenize + stopwords
C = {}  # tokenize + stopwords + stemming
D = {}  # tokenize + stopwords + lemmatize

word_set_A = set()
word_set_B = set()
word_set_C = set()
word_set_D = set()

idx = 0

for text in tokens:
    word_set_A.update(text)
    A[idx] = text

    no_stopwords = [word for word in text if word not in stop_words]
    word_set_B.update(no_stopwords)
    B[idx] = no_stopwords

    stem = list(map(stemmer.stem, no_stopwords))
    word_set_C.update(stem)
    C[idx] = stem

    lem = list(map(wnl().lemmatize, no_stopwords))
    word_set_D.update(lem)
    D[idx] = lem

    idx += 1

print(idx)
A_dt = pd.DataFrame({"tokens" : A.values()})
B_dt = pd.DataFrame({"tokens" : B.values()})
C_dt = pd.DataFrame({"tokens" : C.values()})
D_dt = pd.DataFrame({"tokens" : D.values()})

boolean_A = []
word_set_A = sorted(word_set_A)

for tokens in A_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_A]

    boolean_A.append(vector)


boolean_B = []
word_set_B = sorted(word_set_B)

for tokens in B_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_B]

    boolean_B.append(vector)

boolean_C = []
word_set_C = sorted(word_set_C)

for tokens in C_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_C]

    boolean_C.append(vector)

boolean_D = []
word_set_D = sorted(word_set_D)

for tokens in D_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_D]

    boolean_D.append(vector)




table_A = pd.DataFrame(boolean_A, columns=word_set_A)
table_B = pd.DataFrame(boolean_B, columns=word_set_B)
table_C = pd.DataFrame(boolean_C, columns=word_set_C)
table_D = pd.DataFrame(boolean_D, columns=word_set_D)

## Queries

Q_A = {}
Q_B = {}
Q_C = {}
Q_D = {}

Q_tokens = queries_df.text.apply(nltk.word_tokenize)

idx = 0

for text in Q_tokens:
    Q_A[idx] = text

    no_stopwords = [word for word in text if word not in stop_words]
    Q_B[idx] = no_stopwords

    stem = list(map(stemmer.stem, no_stopwords))
    Q_C[idx] = stem

    lem = list(map(wnl().lemmatize, no_stopwords))
    Q_D[idx] = lem

    idx += 1

print(idx)
Q_A_dt = pd.DataFrame({"tokens" : Q_A.values()})
Q_B_dt = pd.DataFrame({"tokens" : Q_B.values()})
Q_C_dt = pd.DataFrame({"tokens" : Q_C.values()})
Q_D_dt = pd.DataFrame({"tokens" : Q_D.values()})

boolean_Q_A = []
for tokens in Q_A_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_A]

    boolean_Q_A.append(vector)

boolean_Q_B = []
for tokens in Q_B_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_B]

    boolean_Q_B.append(vector)

boolean_Q_C = []
for tokens in Q_C_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_C]

    boolean_Q_C.append(vector)

boolean_Q_D = []
for tokens in Q_D_dt["tokens"]:
    token_set = set(tokens)
    vector = [1 if word in token_set else 0 for word in word_set_D]

    boolean_Q_D.append(vector)

Q_table_A = pd.DataFrame(boolean_Q_A, columns=word_set_A)
Q_table_B = pd.DataFrame(boolean_Q_B, columns=word_set_B)
Q_table_C = pd.DataFrame(boolean_Q_C, columns=word_set_C)
Q_table_D = pd.DataFrame(boolean_Q_D, columns=word_set_D)

# Retrieve docs A

# Docs with querie bits is a subset of doc bits

for query_id, query in Q_table_A.iterrows():
    query_words = query[query == 1].index
    matches = table_A[query_words].eq(1).all(axis=1)
    retrieved_ids = table_A.index[matches]

#TODO Comparar


1400
225
